# 05 - Residual Stock LGD Model

**Completed but Unsold Stock - Australian Bank-Style LGD Framework**

This notebook focuses on residual stock risk after practical completion, where recovery depends on sell-down execution rather than construction completion.

| Step | Description |
|---|---|
| 1 | Generate synthetic residual stock dataset |
| 2 | Build base LGD from sell-down and carrying-cost drivers |
| 3 | Apply stress scenarios for downturn LGD |
| 4 | Produce EAD-weighted LGD outputs |
| 5 | Export interview-ready summary tables |


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.lgd_calculation import exposure_weighted_average
from src.reproducibility import set_global_seed

set_global_seed(47)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.abspath('../outputs')
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)



## Governance Baseline (Pre-Step-3)

- **Fallback hierarchy (standard policy):** use observed workout inputs first, then approved proxy inputs, then conservative fallback with explicit disclosure.
- **Proxy logic (standard policy):** transparent proxy assumptions are used where observed internal workout tapes are unavailable.
- **Discount-rate policy:** `discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)`.
- **Calibration status (standard policy):** this notebook is a portfolio-project proxy baseline and is **not** internally calibrated to real workout history.
- **Output standard:** report `lgd_base`, `lgd_downturn`, `lgd_final`, EAD-weighted outputs, and base recovery-time metric (`time_to_sale_base`).


## Objective
Build an interview-ready residual stock LGD module for completed but unsold inventory with weighted outputs.

## Drivers
- Unsold unit count
- Absorption speed
- Discount-to-clear
- Holding cost and time to sale

## Logic
- Base/downturn/final LGD with product-appropriate discounting and weighted aggregation

## Downturn
- Slower absorption, larger discount-to-clear, and higher carry/discount rates

## Validation
- Weighted scenario/segment outputs and validation checks

## Limitations
- Synthetic portfolio and proxy assumptions; future calibration requires internal workout data


## Why Residual Stock Differs from Development Finance

Residual stock is a **post-completion** problem. Key differences vs development finance:

- Construction execution risk (cost-to-complete, completion uncertainty) is largely behind the project.
- Main risk shifts to **sell-down execution**: unsold units, absorption speed, discount-to-clear, holding cost, and time to sale.
- Recovery path is inventory liquidation over time, not fund-to-complete decisioning.


## 1. Generate Synthetic Residual Stock Dataset


In [ ]:
n_cases = 240
rng = np.random.default_rng(47)

segments = ['metro_apartment', 'suburban_apartment', 'townhouse', 'mixed_stock']
segment_weights = [0.38, 0.27, 0.22, 0.13]

params = {
    'metro_apartment': {'unit_price': (620000, 1150000), 'unsold_units': (15, 120), 'absorption': (3.0, 9.0), 'discount': (0.05, 0.18), 'holding': (0.045, 0.075)},
    'suburban_apartment': {'unit_price': (450000, 780000), 'unsold_units': (12, 90), 'absorption': (2.4, 7.4), 'discount': (0.04, 0.16), 'holding': (0.040, 0.070)},
    'townhouse': {'unit_price': (520000, 920000), 'unsold_units': (8, 60), 'absorption': (1.8, 5.8), 'discount': (0.05, 0.15), 'holding': (0.038, 0.065)},
    'mixed_stock': {'unit_price': (500000, 1100000), 'unsold_units': (10, 95), 'absorption': (2.0, 6.8), 'discount': (0.06, 0.20), 'holding': (0.045, 0.080)},
}

rows = []
for i in range(1, n_cases + 1):
    seg = rng.choice(segments, p=segment_weights)
    p = params[seg]

    unsold_units = int(rng.integers(p['unsold_units'][0], p['unsold_units'][1] + 1))
    avg_unit_price = float(rng.uniform(p['unit_price'][0], p['unit_price'][1]))
    gross_stock_value = unsold_units * avg_unit_price

    absorption_speed_units_per_month = float(rng.uniform(p['absorption'][0], p['absorption'][1]))
    base_time = unsold_units / max(absorption_speed_units_per_month, 0.8)
    time_to_sale_months = float(np.clip(base_time + rng.normal(0.0, 1.3), 3.0, 36.0))

    discount_to_clear = float(rng.uniform(p['discount'][0], p['discount'][1]))
    holding_cost_rate_pa = float(rng.uniform(p['holding'][0], p['holding'][1]))
    selling_cost_rate = float(rng.uniform(0.018, 0.035))

    leverage = float(np.clip(rng.normal(0.68, 0.08), 0.45, 0.88))
    ead = gross_stock_value * leverage * rng.uniform(0.98, 1.03)

    contract_rate_proxy = float(np.clip(rng.normal(0.070, 0.012), 0.045, 0.105))
    cost_of_funds_proxy = float(np.clip(rng.normal(0.054, 0.009), 0.035, 0.085))
    discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)

    rows.append({
        'case_id': f'RS{i:04d}',
        'segment': seg,
        'unsold_units': unsold_units,
        'avg_unit_price': avg_unit_price,
        'gross_stock_value': gross_stock_value,
        'ead': ead,
        'absorption_speed_units_per_month': absorption_speed_units_per_month,
        'time_to_sale_months': time_to_sale_months,
        'discount_to_clear': discount_to_clear,
        'holding_cost_rate_pa': holding_cost_rate_pa,
        'selling_cost_rate': selling_cost_rate,
        'contract_rate_proxy': contract_rate_proxy,
        'cost_of_funds_proxy': cost_of_funds_proxy,
        'discount_rate': discount_rate,
    })

residual = pd.DataFrame(rows)

print('Residual stock cases:', residual.shape)
display(residual['segment'].value_counts())
residual.head()



## 2. Base LGD Calculation


In [ ]:
def compute_residual_stock_lgd(df, speed_multiplier=1.0, discount_addon=0.0, holding_addon=0.0, rate_addon=0.0):
    x = df.copy()

    speed = (x['absorption_speed_units_per_month'] * speed_multiplier).clip(lower=0.8)
    time_to_sale = (x['unsold_units'] / speed).clip(lower=2.0, upper=42.0)

    discount_to_clear = (x['discount_to_clear'] + discount_addon).clip(lower=0.00, upper=0.45)
    holding_cost_rate = (x['holding_cost_rate_pa'] + holding_addon).clip(lower=0.02, upper=0.16)
    discount_rate = (x['discount_rate'] + rate_addon).clip(lower=0.03, upper=0.18)

    gross_sale_proceeds = x['gross_stock_value'] * (1 - discount_to_clear)
    holding_cost = x['gross_stock_value'] * holding_cost_rate * (time_to_sale / 12.0)
    selling_cost = x['gross_stock_value'] * x['selling_cost_rate']

    net_sale_before_discount = (gross_sale_proceeds - holding_cost - selling_cost).clip(lower=0.0)
    discount_factor = (1 + discount_rate) ** (time_to_sale / 12.0)
    net_recovery_pv = net_sale_before_discount / discount_factor

    lgd = ((x['ead'] - net_recovery_pv) / x['ead']).clip(lower=0.00, upper=1.00)

    return lgd, time_to_sale, net_recovery_pv

residual['lgd_base'], residual['time_to_sale_base'], residual['net_recovery_pv_base'] = compute_residual_stock_lgd(residual)

print(f"Portfolio EAD-weighted Base LGD: {exposure_weighted_average(residual, 'lgd_base', 'ead'):.2%}")
display(residual[['lgd_base', 'time_to_sale_base', 'net_recovery_pv_base']].describe().round(4))



## 3. Stress Scenarios, Downturn LGD, and Final Overlay

Stress logic: slower absorption, deeper discount-to-clear, higher carrying cost, and higher discount rate.  
A small explicit final overlay is then applied to downturn LGD so cross-product outputs compare a consistent `lgd_final` metric.


In [ ]:
scenario_specs = [
    {'scenario': 'base', 'speed_multiplier': 1.00, 'discount_addon': 0.00, 'holding_addon': 0.000, 'rate_addon': 0.000},
    {'scenario': 'mild_stress', 'speed_multiplier': 0.82, 'discount_addon': 0.04, 'holding_addon': 0.010, 'rate_addon': 0.005},
    {'scenario': 'severe_stress', 'speed_multiplier': 0.65, 'discount_addon': 0.09, 'holding_addon': 0.022, 'rate_addon': 0.015},
]

scenario_rows = []
for s in scenario_specs:
    lgd_s, time_s, rec_s = compute_residual_stock_lgd(
        residual,
        speed_multiplier=s['speed_multiplier'],
        discount_addon=s['discount_addon'],
        holding_addon=s['holding_addon'],
        rate_addon=s['rate_addon'],
    )

    scenario_name = s['scenario']
    residual[f'lgd_{scenario_name}'] = lgd_s
    residual[f'time_to_sale_{scenario_name}'] = time_s
    residual[f'net_recovery_pv_{scenario_name}'] = rec_s

    overlay_s = (
        0.010
        + 0.015 * ((time_s - 12.0) / 24.0).clip(0.0, 1.0)
        + 0.020 * (residual['discount_to_clear'] + s['discount_addon']).clip(0.0, 0.45)
    ).clip(0.010, 0.070)
    residual[f'final_overlay_addon_{scenario_name}'] = overlay_s
    residual[f'lgd_final_{scenario_name}'] = (lgd_s + overlay_s).clip(0.0, 1.0)

    scenario_rows.append({
        'scenario': scenario_name,
        'ead_weighted_lgd_base': exposure_weighted_average(residual, f'lgd_{scenario_name}', 'ead'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(residual, f'lgd_{scenario_name}', 'ead'),
        'ead_weighted_lgd_final': exposure_weighted_average(residual, f'lgd_final_{scenario_name}', 'ead'),
        'mean_time_to_sale_months': float(time_s.mean()),
    })

scenario_summary = pd.DataFrame(scenario_rows)
scenario_summary['mean_recovery_time_months'] = scenario_summary['mean_time_to_sale_months']
residual['lgd_downturn'] = residual['lgd_severe_stress']
residual['final_overlay_addon'] = residual['final_overlay_addon_severe_stress']
residual['lgd_final'] = residual['lgd_final_severe_stress']

display(scenario_summary.round(4))



## 4. EAD-Weighted LGD Outputs


In [ ]:
segment_rows = []
for seg, grp in residual.groupby('segment', observed=True):
    segment_rows.append({
        'segment': seg,
        'case_count': len(grp),
        'total_ead': grp['ead'].sum(),
        'ead_weighted_lgd_base': exposure_weighted_average(grp, 'lgd_base', 'ead'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(grp, 'lgd_downturn', 'ead'),
        'ead_weighted_lgd_final': exposure_weighted_average(grp, 'lgd_final', 'ead'),
        'mean_unsold_units': grp['unsold_units'].mean(),
        'mean_absorption_speed': grp['absorption_speed_units_per_month'].mean(),
        'mean_discount_to_clear': grp['discount_to_clear'].mean(),
        'mean_holding_cost_rate_pa': grp['holding_cost_rate_pa'].mean(),
        'mean_time_to_sale_months': grp['time_to_sale_base'].mean(),
    })

segment_summary = pd.DataFrame(segment_rows).sort_values('ead_weighted_lgd_final', ascending=False).reset_index(drop=True)
display(segment_summary.round(4))

fig, ax = plt.subplots(figsize=(9, 5))
plot_df = segment_summary.set_index('segment')[['ead_weighted_lgd_base', 'ead_weighted_lgd_downturn', 'ead_weighted_lgd_final']]
plot_df.plot(kind='bar', ax=ax, color=['#4c72b0', '#dd8452', '#c44e52'])
ax.set_title('Residual Stock LGD by Segment (EAD-Weighted Base/Downturn/Final)')
ax.set_ylabel('LGD')
ax.set_xlabel('Segment')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'residual_stock_segment_weighted_lgd.png'), dpi=140, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'residual_stock_weighted_lgd_comparison.png'), dpi=140, bbox_inches='tight')
plt.show()



## 5. Validation and Exports


In [ ]:
validation_checks = pd.DataFrame([
    {'check_name': 'lgd_base_between_0_and_1', 'passed': residual['lgd_base'].between(0, 1).all()},
    {'check_name': 'lgd_downturn_between_0_and_1', 'passed': residual['lgd_downturn'].between(0, 1).all()},
    {'check_name': 'lgd_final_between_0_and_1', 'passed': residual['lgd_final'].between(0, 1).all()},
    {'check_name': 'downturn_not_below_base', 'passed': (residual['lgd_downturn'] >= residual['lgd_base']).all()},
    {'check_name': 'final_not_below_downturn', 'passed': (residual['lgd_final'] >= residual['lgd_downturn']).all()},
])
display(validation_checks)

residual.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'residual_stock_loan_level_output.csv'), index=False)
segment_summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'residual_stock_segment_summary.csv'), index=False)
scenario_summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'residual_stock_scenario_summary.csv'), index=False)
validation_checks.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'residual_stock_validation_checks.csv'), index=False)

print('Saved residual stock outputs:')
print('- residual_stock_loan_level_output.csv')
print('- residual_stock_segment_summary.csv')
print('- residual_stock_scenario_summary.csv')
print('- residual_stock_validation_checks.csv')



## Assumptions and Limitations

- Residual stock data is synthetic and designed for transparent portfolio demonstration.
- Absorption speed, discount-to-clear, holding cost, and time-to-sale are proxy variables in lieu of internal realised workout tapes.
- Stress scenario parameters are pragmatic interview-grade overlays and should be recalibrated using observed stock liquidation history before production use.


---

## APS 113 Calibration Layer — Residual Stock

> **SYNTHETIC HISTORICAL CALIBRATION DATA — FOR DEMONSTRATION ONLY**
>
> This section adds a full APS 113-aligned calibration loop on top of the
> proxy baseline above. Workout data is synthetically generated (2014-2024,
> 10-year window) to demonstrate methodology; real production calibration
> requires an internal workout tape.

### Calibration Pipeline (APS 113 s.32-68)

| Step | Function | APS 113 |
|------|----------|---------|
| 1 | Load/generate historical workout data | s.44, Att A |
| 2 | `compute_realised_lgd()` — LIP costs, cure leg | s.32-34 |
| 3 | `classify_economic_regime()` + `assign_regime_to_workouts()` | s.43, s.46 |
| 4 | `segment_lgd()` — product-specific segment keys | s.52 |
| 5 | `compute_long_run_lgd()` — vintage-EWA method | s.43-44 |
| 6 | `compare_model_vs_actual()` — proxy vs realised | s.60-62 |
| 7 | `apply_downturn_overlay()` + `apply_correlation_adjustment()` | s.46-50, s.55-57 |
| 8 | `MoCRegister` + `apply_moc()` — 5 APS 113 s.65 sources | s.63-65 |
| 9 | `apply_regulatory_floor()` — 30% per APS 113 s.58 | s.58 |
| 10 | Export 9 CSV outputs | — |
| 11 | `run_full_validation_suite()` — Gini, HL, PSI, OOT | s.66-68 |

**Correct APS 113 order:** LR-LGD → downturn overlay → correlation adj →
MoC → floor (MoC applied to downturn LGD, not long-run LGD, per s.63).


In [ ]:
# APS 113 Calibration Layer — imports and configuration
import os, sys
from pathlib import Path
sys.path.insert(0, os.path.abspath('..'))

from src.calibration_utils import (
    compute_realised_lgd,
    segment_lgd,
    compute_long_run_lgd,
    compare_model_vs_actual,
    classify_economic_regime,
    assign_regime_to_workouts,
    apply_downturn_overlay,
    apply_correlation_adjustment,
    build_lgd_pd_annual_series,
    estimate_lgd_pd_correlation,
    MoCRegister,
    apply_moc,
    apply_regulatory_floor,
    run_full_validation_suite,
    generate_compliance_map,
    export_compliance_map,
    CALIBRATION_STEP_ORDER,
)
from src.generators import GENERATOR_MAP, generate_all_historical_workouts

PRODUCT = "residual_stock"
SEGMENT_KEYS = ['market_depth_proxy', 'stock_age_band']
MODEL_LGD_COL = "lgd_final"

HISTORY_DIR = Path('..') / 'data' / 'generated' / 'historical'
OUTPUTS_DIR = Path('..') / 'outputs' / 'tables'
UPSTREAM_PARQUET = Path('..') / 'data' / 'exports' / 'macro_regime_flags.parquet'

HISTORY_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Product: {PRODUCT}")
print(f"Segment keys: {SEGMENT_KEYS}")
print(f"APS 113 calibration pipeline — step order:")
for step, fn, ref in CALIBRATION_STEP_ORDER:
    print(f"  Step {step:>2}: {fn:<35} {ref}")


In [ ]:
# ── STEP 1: Load or generate historical workout data (APS 113 s.44, Att A) ─
parquet_path = HISTORY_DIR / f"{PRODUCT}_workouts.parquet"

if parquet_path.exists():
    cal_loans = pd.read_parquet(parquet_path)
    # cashflows stored alongside or re-generated
    try:
        cal_cashflows = pd.read_parquet(
            HISTORY_DIR / f"{PRODUCT}_cashflows.parquet"
        )
    except FileNotFoundError:
        cal_cashflows = None
    print(f"Loaded {len(cal_loans):,} historical workout loans from Parquet")
else:
    print(f"Parquet not found — generating synthetic workout data for {PRODUCT}")
    results = generate_all_historical_workouts(
        seed=42, output_dir=HISTORY_DIR, write_parquet=True, products=[PRODUCT]
    )
    cal_loans = results[PRODUCT]["loans"]
    cal_cashflows = results[PRODUCT].get("cashflows")
    print(f"Generated {len(cal_loans):,} synthetic workout loans (2014-2024)")

print(f"Date range: {cal_loans['default_date'].min()} to {cal_loans['default_date'].max()}")
print(f"Columns: {list(cal_loans.columns)}")

# ── STEP 2: Compute realised LGD (APS 113 s.32-34) ─────────────────────────
# LIP costs (Loss Identification Period, ~90 days) auto-detected if cashflows available
lgd_df = compute_realised_lgd(
    loans=cal_loans,
    cashflows=cal_cashflows,
    ead_col="ead_at_default",
    recovery_col="recovery_amount",
    cost_col="direct_costs",
    lip_window_days=90,
)
print(f"\nStep 2: Realised LGD computed")
print(f"  EAD-weighted realised LGD: {(lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum():.2%}")
display(lgd_df[['realised_lgd', 'ead_at_default']].describe().round(4))

# ── STEP 3: Economic regime classification (APS 113 s.43, s.46-50) ─────────
upstream_path = str(UPSTREAM_PARQUET) if UPSTREAM_PARQUET.exists() else None
regime_df = classify_economic_regime(
    upstream_parquet_path=upstream_path,
    method="upstream_first",
)
print(f"\nStep 3: Economic regimes classified")
print(f"  Data source: {regime_df['data_source'].iloc[0]}")
display(regime_df[['year', 'regime', 'is_downturn_period', 'data_source']].head(12))

lgd_df = assign_regime_to_workouts(lgd_df, regime_df)
downturn_pct = lgd_df.get('is_downturn_period', pd.Series([False])).mean()
print(f"  Downturn observations: {downturn_pct:.1%} of portfolio")

# ── STEP 4: Segment by product-specific keys (APS 113 s.52) ────────────────
segmented_df = segment_lgd(lgd_df, SEGMENT_KEYS)
low_count = segmented_df[segmented_df.get('segment_flag', '') == 'low_count'] if 'segment_flag' in segmented_df.columns else pd.DataFrame()
print(f"\nStep 4: Segmentation complete")
print(f"  Segments: {segmented_df.groupby(SEGMENT_KEYS, observed=True).ngroups}")
if not low_count.empty:
    print(f"  Low-count segments flagged (<20 obs): {len(low_count)}")

# ── STEP 5: Long-run LGD — vintage-EWA (APS 113 s.43-44) ─────────────────
lr_lgd_df = compute_long_run_lgd(
    segmented_df,
    segment_keys=SEGMENT_KEYS,
    method="vintage_ewa",
)
print(f"\nStep 5: Long-run LGD by segment (vintage-EWA)")
display(lr_lgd_df.round(4))
lr_lgd_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_long_run_lgd_by_segment.csv", index=False)

# ── STEP 6: Compare model vs actual (APS 113 s.60-62) ──────────────────────
# 'model_lgd' here = proxy model LGD from the section above (lgd_final if present)
if MODEL_LGD_COL in cal_loans.columns:
    compare_input = lgd_df.merge(
        cal_loans[['loan_id', MODEL_LGD_COL] if 'loan_id' in cal_loans.columns else [MODEL_LGD_COL]],
        left_index=True, right_index=True, how='left',
    ) if MODEL_LGD_COL not in lgd_df.columns else lgd_df
else:
    compare_input = lgd_df.copy()
    compare_input['model_lgd'] = lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else 0.25

model_col_to_use = MODEL_LGD_COL if MODEL_LGD_COL in compare_input.columns else 'model_lgd'
mva_df = compare_model_vs_actual(
    compare_input,
    model_lgd_col=model_col_to_use,
    segment_keys=SEGMENT_KEYS,
)
print(f"\nStep 6: Model vs actual comparison")
display(mva_df.round(4))
mva_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_model_vs_actual_comparison.csv", index=False)

# ── STEP 7: Downturn overlay + Frye-Jacobs correlation adj (s.46-50, s.55-57)
# Reuses apply_downturn_overlay from existing lgd_calculation.py
dt_lgd = apply_downturn_overlay(segmented_df, product=PRODUCT)
print(f"\nStep 7: Downturn overlay applied")
if 'lgd_downturn' in dt_lgd.columns:
    ewa_dt = (dt_lgd['lgd_downturn'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
    ewa_lr = (dt_lgd['realised_lgd'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
    print(f"  EWA Long-run LGD:  {ewa_lr:.2%}")
    print(f"  EWA Downturn LGD:  {ewa_dt:.2%}")
    downturn_col = 'lgd_downturn'
else:
    dt_lgd['lgd_downturn'] = dt_lgd['realised_lgd'] * 1.15  # fallback scalar
    downturn_col = 'lgd_downturn'

# Frye-Jacobs correlation adjustment (APS 113 s.55-57)
try:
    lgd_ts, pd_ts = build_lgd_pd_annual_series(dt_lgd)
    macro_for_corr = regime_df.rename(columns={'gdp_growth': 'gdp_growth_yoy'})
    corr_result = estimate_lgd_pd_correlation(lgd_ts, pd_ts, macro_for_corr)
    dt_lgd['lgd_downturn_corr_adj'] = apply_correlation_adjustment(
        dt_lgd[downturn_col], corr_result['rho'], corr_result['macro_shock_std']
    )
    downturn_col = 'lgd_downturn_corr_adj'
    print(f"  Frye-Jacobs rho={corr_result['rho']:.3f}, adj factor={corr_result['lgd_dt_adjustment_factor']:.3f}")
except Exception as exc:
    print(f"  Frye-Jacobs skipped: {exc}")

dt_lgd.to_csv(OUTPUTS_DIR / f"{PRODUCT}_downturn_lgd_by_segment.csv", index=False)

# ── STEP 8: MoC register + apply MoC (AFTER downturn — APS 113 s.63-65) ───
# Determine regime data source for MoC model_approximation component
data_src = regime_df['data_source'].iloc[0] if 'data_source' in regime_df.columns else 'synthetic'
n_downturn_yrs = int(regime_df['is_downturn_period'].sum()) if 'is_downturn_period' in regime_df.columns else 0

psi_approx = 0.05  # placeholder — full PSI computed in Step 11
bias_approx = float(mva_df['bias'].abs().mean()) if 'bias' in mva_df.columns else 0.02

moc_register = MoCRegister(product=PRODUCT, regime_data_source=data_src)
moc_df = moc_register.build_moc_register(
    segment_df=segmented_df,
    segment_keys=SEGMENT_KEYS,
    n_downturn_vintages=n_downturn_yrs,
    psi_value=psi_approx,
    backtesting_bias=bias_approx,
)
print(f"\nStep 8: MoC register built")
display(moc_df.round(4))
moc_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_moc_register.csv", index=False)

lgd_with_moc = apply_moc(dt_lgd[downturn_col], moc_df, segment_col=SEGMENT_KEYS[0] if SEGMENT_KEYS else None)
dt_lgd['lgd_with_moc'] = lgd_with_moc
moc_ewa = (lgd_with_moc * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
print(f"  EWA LGD after MoC: {moc_ewa:.2%}")

# ── STEP 9: Regulatory floors (APS 113 s.58) ──────────────────────────────
dt_lgd['lgd_final_calibrated'] = apply_regulatory_floor(dt_lgd['lgd_with_moc'], product=PRODUCT)
final_ewa = (dt_lgd['lgd_final_calibrated'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
floor_binding_pct = (dt_lgd['lgd_final_calibrated'] > dt_lgd['lgd_with_moc']).mean()
print(f"\nStep 9: Regulatory floor applied")
print(f"  EWA Final Calibrated LGD: {final_ewa:.2%}")
print(f"  Floor binding for: {floor_binding_pct:.1%} of loans")

dt_lgd.to_csv(OUTPUTS_DIR / f"{PRODUCT}_final_calibrated_lgd.csv", index=False)

# ── STEP 10: Export all outputs ────────────────────────────────────────────
# Already exported: long_run_lgd_by_segment, model_vs_actual, downturn_lgd, moc_register, final_calibrated_lgd
# Export remaining:
lgd_df[['realised_lgd', 'ead_at_default']].assign(product=PRODUCT).to_csv(
    OUTPUTS_DIR / f"{PRODUCT}_historical_workouts.csv", index=False
)
regime_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_regime_classification.csv", index=False)

# Calibration adjustments summary
cal_adj_summary = pd.DataFrame({
    'product': [PRODUCT],
    'ewa_realised_lgd': [(lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum()],
    'ewa_long_run_lgd': [lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else None],
    'ewa_downturn_lgd': [(dt_lgd.get('lgd_downturn', dt_lgd['realised_lgd']) * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()],
    'ewa_lgd_with_moc': [(dt_lgd['lgd_with_moc'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()],
    'ewa_lgd_final': [final_ewa],
    'floor_binding_pct': [floor_binding_pct],
    'regime_data_source': [data_src],
    'n_loans': [len(lgd_df)],
    'calibration_date': [pd.Timestamp.today().date()],
})
cal_adj_summary.to_csv(OUTPUTS_DIR / f"{PRODUCT}_calibration_adjustments.csv", index=False)
print(f"\nStep 10: All outputs exported to {OUTPUTS_DIR}")

# ── STEP 11: Full validation suite (APS 113 s.66-68) ──────────────────────
print(f"\nStep 11: Running full validation suite...")
try:
    val_results = run_full_validation_suite(
        loans=dt_lgd,
        predicted_col='lgd_final_calibrated',
        actual_col='realised_lgd',
        segment_col=SEGMENT_KEYS[0] if SEGMENT_KEYS else None,
        date_col='default_date' if 'default_date' in dt_lgd.columns else None,
        product=PRODUCT,
    )
    print(f"  Gini: {val_results.get('gini', 'n/a')}")
    print(f"  Calibration ratio: {val_results.get('calibration_ratio', 'n/a')}")
    print(f"  PSI: {val_results.get('psi', 'n/a')}")
    if 'summary_table' in val_results:
        val_results['summary_table'].to_csv(
            OUTPUTS_DIR / f"{PRODUCT}_validation_report.csv", index=False
        )
    if 'backtest_results' in val_results:
        val_results['backtest_results'].to_csv(
            OUTPUTS_DIR / f"{PRODUCT}_backtest_results.csv", index=False
        )
except Exception as exc:
    print(f"  Validation suite error (non-fatal): {exc}")


In [ ]:
# ── Calibration summary waterfall ──────────────────────────────────────────
import matplotlib.pyplot as plt

try:
    stages = {
        'Realised LGD\n(2014-2024)': (lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum(),
        'Long-run LGD\n(vintage-EWA)': lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else None,
        'Downturn LGD': (dt_lgd.get(downturn_col, dt_lgd['realised_lgd']) * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum(),
        '+ MoC': (dt_lgd['lgd_with_moc'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum(),
        'Final\n(Floor Applied)': final_ewa,
    }
    labels = [k for k, v in stages.items() if v is not None]
    values = [v for v in stages.values() if v is not None]
    colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c', '#8e44ad'][:len(labels)]

    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.bar(labels, values, color=colors, edgecolor='white', width=0.6)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax.set_ylabel('EAD-Weighted LGD')
    ax.set_title(f'APS 113 Calibration Waterfall — Residual Stock')
    ax.set_ylim(0, max(values) * 1.35)
    ax.axhline(values[-1], color='black', ls=':', lw=1, label=f'Final: {values[-1]:.1%}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(
        Path('..') / 'outputs' / 'figures' / f'residual_stock_calibration_waterfall.png',
        dpi=150, bbox_inches='tight',
    )
    plt.show()
except Exception as exc:
    print(f"Waterfall chart error (non-fatal): {exc}")

# ── APS 113 compliance snapshot ─────────────────────────────────────────────
compliance_df = generate_compliance_map(
    calibration_results={PRODUCT: {'long_run_lgd_by_segment': True, 'calibration_steps': True}},
    moc_registers={PRODUCT: moc_df},
    regime_data_source=data_src,
    products=[PRODUCT],
)
print("\n=== APS 113 Compliance Snapshot ===")
display(compliance_df[['section_ref', 'requirement', 'status', 'reviewer_note']].set_index('section_ref'))
export_compliance_map(compliance_df, OUTPUTS_DIR / f"residual_stock_aps113_compliance.csv")

# Final summary
print("\n=== Calibration Summary ===")
display(cal_adj_summary.round(4))
print(f"\nAll calibration outputs in: {OUTPUTS_DIR}")
print(f"SYNTHETIC HISTORICAL CALIBRATION DATA — FOR DEMONSTRATION ONLY")
